# B01. Configuration

Every world and system in TidalPy is defined by a TOML configuration file, and the package's own defaults live in one global configuration file. This demo shows where these files live, what they contain, how to load one, edit it in a live session, and build a world from it, how a build checks a config, and how the global configuration fills in what a world file leaves out.

## Where Configurations Live

TidalPy ships a set of example worlds. `available_worlds()` lists them by name and `get_worlds_dir()` gives the per-user directory they are installed into: `<Documents>/TidalPy/<version>/Worlds`, or the same folders under the `TIDALPY_DATA_DIR` environment variable when it is set. That directory also holds multi-world system configs, listed by `available_systems()`. The named materials a layer can be built from (MatPack) are installed beside it and listed by `available_materials()`. Every config carries a `schema_version` that tells TidalPy how to read it.

The first `build_world("<name>")` copies the bundled files into this directory, and every later build reads the copy, so editing `<data dir>/Worlds/<name>.toml` changes that world for every session. A newer TidalPy never overwrites a copy: when a copy differs from the packaged file, TidalPy warns once and keeps using the copy, and `install_worldpack(force=True)` restores the packaged files.

In [1]:
from pathlib import Path

from TidalPy.Structures.configs import (
    available_worlds,
    available_systems,
    get_worlds_dir,
    resolve_world_path,
    load_toml,
    validate_world_config,
    build_world,
    SCHEMA_VERSION,
)
from TidalPy.Material import available_materials

# The start of the path depends on the machine, so only the part TidalPy adds is printed
worlds_dir = Path(get_worlds_dir())
print("current schema version:", SCHEMA_VERSION)
print("worlds directory:", "<data dir>/" + "/".join(worlds_dir.parts[-2:]))
print("worlds available:", available_worlds())
print("systems available:", available_systems())
print("materials available:", available_materials())

current schema version: 0.2.0
worlds directory: <data dir>/0.8.X/Worlds
worlds available: ['charon', 'earth_prem', 'earth_prem_q', 'earth_simple', 'earth_thermal', 'europa', 'europa_dynamic', 'io', 'jupiter', 'jupiter_simple', 'luna', 'luna_dynamic', 'mercury', 'mercury_dynamic', 'neptune', 'pluto', 'pluto_dynamic', 'sol', 'trappist1', 'trappist1b', 'trappist1c', 'trappist1d', 'trappist1e', 'trappist1f', 'trappist1g', 'trappist1h', 'triton']
systems available: ['sol_system']
materials available: ['ammonia_water', 'basalt', 'brine', 'chondrite', 'felsic_crust', 'giant_core', 'h2_he_molecular', 'h_he_metallic', 'ice_ih', 'ice_iii', 'ice_v', 'ice_vi', 'ice_vii', 'iron', 'iron_sulfide', 'liquid_iron', 'lower_mantle', 'methane_clathrate', 'nitrogen_ice', 'olivine', 'peridotite', 'serpentinite', 'simple_gas', 'simple_ice', 'simple_iron_core', 'simple_liquid_iron', 'simple_rock', 'simple_water', 'water']


## Config File Contents

`resolve_world_path` converts a world name into the path of its TOML file. The three-layer `earth_simple` file holds a name, bulk properties (radius, mass, spin), an `[eos_solver]` table, and one table per layer. Each layer table sets its outer radius, its temperature, its physics switches (`use_tides`, `use_melting`, `use_heating`, ...), and its material. Here every material is written out in full: a solid phase (equation of state, shear modulus, viscosity) for the inner core, a liquid-only phase for the outer core, and both phases with melting curves for the mantle. A layer may instead name a MatPack material (`material = "peridotite"`), or take one as a `preset` and override some of its values. The mantle also names its rheology, cooling, and radiogenic heating models.

In [2]:
earth_path = Path(resolve_world_path("earth_simple"))
print(earth_path.name)
print()
print(earth_path.read_text())

earth_simple.toml

# Earth-Simple: a three-layer Earth for the Structures world builder, and the worked example of its
# schema. A solid inner core and a static liquid outer core under a tidally active rocky mantle, with the
# published mass, moment of inertia, and degree-2 Love number reproduced; earth_prem is the seismic
# profile for work that needs one.
#
# Each layer states its material in full: a solid or a liquid phase, each with an equation of state and,
# for a solid, a shear modulus and a viscosity law. A layer may instead name a MatPack material (material =
# "peridotite"), or a preset with overrides (TidalPy.Material.available_materials() lists them). A layer
# with a liquid-only material, like the outer core, is a liquid. Each solid layer names its rheology.
# Layers are built inner-to-outer: the inner radius is derived from the previous layer (0 for the
# innermost), and each layer's outer radius is set by exactly one of radius_outer_m, radius_fraction
# (* world radius),

## Loading a Config as a Dictionary

`load_toml` reads a TOML file into a nested dictionary. This is the form you edit in a live session.

In [3]:
import pprint

config = load_toml(earth_path)
pprint.pp(config)

{'schema_version': '0.2.0',
 'name': 'Earth-Simple',
 'type': 'terrestrial',
 'radius_m': 6371000.0,
 'mass_kg': 5.972e+24,
 'spin_frequency_rad_s': 7.292e-05,
 'eos_solver': {'solve_temperature': False},
 'layers': {'inner_core': {'layer_index': 0,
                           'radius_outer_m': 1221500.0,
                           'use_tides': True,
                           'temperature_k': 5500.0,
                           'use_heating': True,
                           'material': {'solid': {'thermal_conductivity_w_mk': 7.95,
                                                  'heat_capacity_j_kgk': 840.0,
                                                  'eos': {'model': 'constant',
                                                          'reference_density_kg_m3': 12900.0,
                                                          'bulk_modulus_pa': 160000000000.0,
                                                          'thermal_expansion_1_k': 1.2e-05},
                        

## Building a World from a Name, a Path, or a Dictionary

`build_world` accepts a bundled world name, the path to a TOML file, or a config dictionary. All three return the same kind of world object.

In [4]:
world_from_name = build_world("earth_simple")
world_from_path = build_world(earth_path)
world_from_dict = build_world(config)

for world in (world_from_name, world_from_path, world_from_dict):
    print(repr(world))

TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)
TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)
TerrestrialWorld('Earth-Simple', radius_km=6371, mass_kg=5.9720e+24, num_layers=3)


## Editing a Config in a Live Session

The config is a dictionary, so it can be edited in memory and built into a new world without changing any file. The edit below enlarges the iron core and guesses a heavier mass for the result. `validate_world_config` checks the edited dictionary against the schema before the build.

The stated `mass_kg` is not an input to the interior structure: `solve_eos` integrates the layers' materials outward from the center and finds the mass they hold. When the two differ by more than 1 percent, TidalPy warns once per world, as it does below. The Love numbers and tides follow the solved structure, while a `System` orbit, `calc_mean_density`, and `calc_surface_gravity` use the stated mass.

In [5]:
import copy

edited = copy.deepcopy(config)
edited["name"] = "Earth-Heavy"
edited["layers"]["outer_core"]["radius_outer_m"] = 4.0e6  # larger core
edited["mass_kg"] = 7.0e24  # a guess at the heavier world's mass

validate_world_config(edited)  # raises if the edit broke the schema
heavy = build_world(edited)

world_from_name.solve_eos()
heavy.solve_eos()  # the solved mass is not the guess, so this warns

print(f"{'world':14} {'core R (m)':>11} {'stated mass (kg)':>17} {'solved mass (kg)':>17} {'C/MR^2':>7}")
for world, world_config in ((world_from_name, config), (heavy, edited)):
    core_radius = world_config["layers"]["outer_core"]["radius_outer_m"]
    print(f"{world.name:14} {core_radius:11.3e} {world.mass:17.3e} {world.planet_mass_eos:17.3e} "
          f"{world.moment_of_inertia_factor:7.4f}")

[TidalPy] [warning] TidalPy: world 'Earth-Heavy' solved to a mass of 6.601261e+24 kg, 5.7 percent off its stated mass of 7.000000e+24 kg. Its Love numbers, tides, and moment of inertia follow the solved structure, while its orbit in a System, calc_surface_gravity, and calc_mean_density use the stated mass. Adjust the layers' materials or radii, or the stated mass, so the two agree. Shown once per world.


world           core R (m)  stated mass (kg)  solved mass (kg)  C/MR^2
Earth-Simple     3.480e+06         5.972e+24         5.972e+24  0.3307
Earth-Heavy      4.000e+06         7.000e+24         6.601e+24  0.3230


Earth-Simple's materials were fitted to its mass, so its stated and solved masses agree. The larger core holds 6.60e24 kg, not the 7.0e24 kg guessed. Writing the solved mass back into the config makes the two agree, and the rebuilt world solves without a warning.

In [6]:
edited["mass_kg"] = heavy.planet_mass_eos
heavy = build_world(edited)
heavy.solve_eos()

print(f"stated mass: {heavy.mass:.3e} kg, solved mass: {heavy.planet_mass_eos:.3e} kg")
print(f"mean density: {heavy.calc_mean_density():.1f} kg/m^3 (Earth-Simple: {world_from_name.calc_mean_density():.1f})")

stated mass: 6.601e+24 kg, solved mass: 6.601e+24 kg
mean density: 6094.2 kg/m^3 (Earth-Simple: 5513.3)


## A Config That Fails Validation

`validate_world_config` and `build_world` refuse a config they cannot read, naming the problem, so a mistake never builds a world silently. Two common mistakes are a misspelled key, for which the message suggests the closest key a layer reads, and a layer given more than one outer radius.

In [7]:
misspelled = copy.deepcopy(config)
misspelled["layers"]["mantle"]["temprature_k"] = misspelled["layers"]["mantle"].pop("temperature_k")

two_radii = copy.deepcopy(config)
two_radii["layers"]["mantle"]["radius_outer_m"] = 6.371e6  # the layer already sets radius_fraction

for broken_config in (misspelled, two_radii):
    try:
        build_world(broken_config)
    except ValueError as error:
        print("ValueError:", error)
        print()

ValueError: Unexpected key 'temprature_k' (did you mean 'temperature_k'?) on layer 'mantle'. Allowed keys: ['is_incompressible', 'is_static', 'is_volume_fixed', 'mass_kg', 'state', 'temperature_k', 'tidal_scale', 'use_heating', 'use_melt_density', 'use_melting', 'use_pressure_melting', 'use_thermal_expansion', 'use_tides'].

ValueError: Layer 'mantle' specifies multiple outer-radius keys ['radius_outer_m', 'radius_fraction']; use exactly one of ('radius_outer_m', 'radius_fraction', 'volume_fraction').



## Defaults and Materials

`build_world` fills in every value a world or layer table leaves out. A layer's physical properties come from its material, either the full table written in the file, as in `earth_simple`, or a named MatPack material. A layer that names no material takes the one in the `[layers]` table of TidalPy's global configuration (`TidalPy_Configs.toml` in the data directory, loaded as `TidalPy.config`), which is `simple_rock` unless the user changes it. The `[worlds]` and `[tides]` tables of the same file supply the world-level defaults, such as the albedo and the tide model of each world type.

The compact config below has two layers: a core of `simple_iron_core` and a mantle that names no material. `get_config_dict` reports the built world with every default filled in.

In [8]:
import TidalPy

print("global [layers] table:", TidalPy.config["layers"])
print("default tide model per type:", TidalPy.config["tides"]["default_model"])

compact = {
    "name": "Compact-World",
    "type": "terrestrial",
    "radius_m": 3.0e6,
    "mass_kg": 4.4e23,
    "layers": {
        "core": {"radius_fraction": 0.5, "material": "simple_iron_core"},
        "mantle": {"radius_fraction": 1.0},  # no material: the global [layers] default is used
    },
}
validate_world_config(compact)
compact_world = build_world(compact)

built = compact_world.get_config_dict()
print("tide model of the built world:", built["tides"]["global_tidal_model"])
print()
print(f"{'layer':8} {'density (kg/m^3)':>17} {'shear modulus (Pa)':>19} {'viscosity (Pa s)':>17} {'rheology':>9}")
for layer_name in ("core", "mantle"):
    solid = built["layers"][layer_name]["material"]["solid"]
    print(f"{layer_name:8} {solid['eos']['reference_density_kg_m3']:17.1f} "
          f"{solid['shear_modulus']['shear_modulus_pa']:19.2e} "
          f"{solid['shear_viscosity']['reference_viscosity_pas']:17.1e} {solid['shear_rheology']['model']:>9}")

global [layers] table: {'material': 'simple_rock'}
default tide model per type: {'star': 'fixed_q', 'gasgiant': 'fixed_dt', 'terrestrial': 'rheology', 'layered': 'rheology'}
tide model of the built world: rheology

layer     density (kg/m^3)  shear modulus (Pa)  viscosity (Pa s)  rheology
core                8000.0            8.00e+10           1.0e+20   maxwell
mantle              3300.0            6.00e+10           1.0e+21   andrade


## Changing the Global Configuration

Editing `TidalPy_Configs.toml` changes the defaults for every later session. To change them for the current session only, `TidalPy.reinit(provided_config=...)` merges a dictionary or a TOML file over the loaded configuration, and `TidalPy.reinit(provided_config="default")` returns to the file's settings. `TidalPy.save_config(path)` writes the configuration in effect, headed by the package versions in use. Kept next to a world or system TOML, it lets someone with the same TidalPy version reproduce a result.

Below, the session's default material becomes `basalt`, so the same compact config now builds a basalt mantle.

In [9]:
import tempfile

TidalPy.reinit(provided_config={"layers": {"material": "basalt"}})  # this session only
basalt_world = build_world(compact)
mantle_eos = basalt_world.get_config_dict()["layers"]["mantle"]["material"]["solid"]["eos"]
print("mantle material:", mantle_eos["model"], "law,", mantle_eos["reference_density_kg_m3"], "kg/m^3")

with tempfile.TemporaryDirectory() as temp_dir:
    saved_path = TidalPy.save_config(Path(temp_dir) / "my_run_config.toml")
    print()
    print("\n".join(Path(saved_path).read_text().splitlines()[:6]))  # the version header of the saved file

TidalPy.reinit(provided_config="default")  # back to the settings in TidalPy_Configs.toml
print()
print("global [layers] table:", TidalPy.config["layers"])

mantle material: birch_murnaghan law, 3030.0 kg/m^3

# =====================================================================================================================
#  TidalPy Configurations
#  TidalPy version: 0.8.0
#  SciPy version: 1.18.0
#  CyRK version: 0.19.3
# =====================================================================================================================

global [layers] table: {'material': 'simple_rock'}
